# Implement Logistic Regression with Gradient Descent — Solution

**Difficulty**: 🟢 Easy

**Companies**: Google, Meta, Amazon, General FAANG

---

### Problem Statement

Logistic regression is the foundation of many classification systems. Under the hood, it applies a **sigmoid** function to a linear model and optimizes using **binary cross-entropy loss** with gradient descent.

Your task is to implement logistic regression **from scratch** using manual gradient computation — no `torch.autograd`, no `loss.backward()`. You will compute the gradients of the binary cross-entropy loss with respect to the weights and bias analytically and apply gradient updates manually.

---

### Requirements

1. **Sigmoid Function** — Implement `sigmoid(z) = 1 / (1 + exp(-z))`.
2. **Binary Cross-Entropy Loss** — Implement `BCE = -mean(y * log(p) + (1-y) * log(1-p))`.
3. **Manual Gradients** — Compute `dw = (1/N) * X^T @ (p - y)` and `db = (1/N) * sum(p - y)`.
4. **Gradient Descent** — Update weights and bias using `w -= lr * dw`, `b -= lr * db`.
5. **Training Loop** — Train for a fixed number of epochs, tracking loss at each step.

---

### Constraints

- ✅ Use only PyTorch tensor operations.
- ✅ Must compute gradients manually (analytically), not via autograd.
- ✅ Loss should decrease over training.
- ✅ Final accuracy should be >85% on the synthetic data.
- ❌ Do **not** use `torch.autograd`, `loss.backward()`, or any optimizer from `torch.optim`.

---

<details>
  <summary>💡 Hint</summary>

  - The gradient of BCE loss w.r.t. the linear output `z = Xw + b` simplifies to `(p - y)` where `p = sigmoid(z)`. This is a well-known result.
  - The gradient w.r.t. weights is `dL/dw = (1/N) * X^T @ (p - y)` and w.r.t. bias is `dL/db = (1/N) * sum(p - y)`.
  - Add a small epsilon (e.g., `1e-7`) inside `log()` to avoid `log(0)`.
  - Start with a learning rate around 0.1 and train for 1000 epochs.

</details>

---

In [ ]:
import torch

In [ ]:
# Generate linearly separable 2D data
torch.manual_seed(42)

n_samples = 200

# Class 0: centered at (-2, -2)
X0 = torch.randn(n_samples // 2, 2) * 0.8 + torch.tensor([-2.0, -2.0])
y0 = torch.zeros(n_samples // 2)

# Class 1: centered at (2, 2)
X1 = torch.randn(n_samples // 2, 2) * 0.8 + torch.tensor([2.0, 2.0])
y1 = torch.ones(n_samples // 2)

X = torch.cat([X0, X1], dim=0)  # (200, 2)
y = torch.cat([y0, y1], dim=0)  # (200,)

# Shuffle
perm = torch.randperm(n_samples)
X, y = X[perm], y[perm]

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Class distribution: {y.sum().item():.0f} positive, {(1 - y).sum().item():.0f} negative")

In [ ]:
def sigmoid(z: torch.Tensor) -> torch.Tensor:
    """Compute the sigmoid function."""
    return 1.0 / (1.0 + torch.exp(-z))


def bce_loss(y_true: torch.Tensor, y_pred: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """Compute binary cross-entropy loss."""
    return -torch.mean(
        y_true * torch.log(y_pred + eps) + (1 - y_true) * torch.log(1 - y_pred + eps)
    )


def train_logistic_regression(X: torch.Tensor, y: torch.Tensor,
                               lr: float = 0.1, n_epochs: int = 1000):
    """
    Train logistic regression with manual gradient descent.

    Args:
        X (Tensor): Features of shape (N, D).
        y (Tensor): Binary labels of shape (N,).
        lr (float): Learning rate.
        n_epochs (int): Number of training epochs.

    Returns:
        w (Tensor): Learned weights of shape (D,).
        b (Tensor): Learned bias (scalar).
        losses (list): Loss at each epoch.
    """
    N, D = X.shape

    # Initialize weights and bias to zeros
    w = torch.zeros(D)
    b = torch.zeros(1)

    losses = []

    for epoch in range(n_epochs):
        # Step 1: Compute linear output
        z = X @ w + b  # (N,)

        # Step 2: Apply sigmoid
        p = sigmoid(z)  # (N,)

        # Step 3: Compute loss
        loss = bce_loss(y, p)
        losses.append(loss.item())

        # Step 4: Compute gradients manually
        error = p - y  # (N,)
        dw = (1.0 / N) * (X.T @ error)  # (D,)
        db = (1.0 / N) * error.sum()  # scalar

        # Step 5: Update weights and bias
        w = w - lr * dw
        b = b - lr * db

        if epoch % 200 == 0:
            print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f}")

    return w, b, losses

In [ ]:
# Validation
w, b, losses = train_logistic_regression(X, y, lr=0.1, n_epochs=1000)

# Check loss decreased
print(f"\nInitial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss did not decrease during training!"
print("Loss decrease test PASSED\n")

# Compute accuracy
with torch.no_grad():
    probs = sigmoid(X @ w + b)
    preds = (probs >= 0.5).float()
    accuracy = (preds == y).float().mean().item()

print(f"Accuracy: {accuracy:.2%}")
print(f"Learned weights: {w}")
print(f"Learned bias: {b.item():.4f}")

assert accuracy > 0.85, f"Accuracy too low: {accuracy:.2%}. Expected >85%."
print("\nAll tests passed!")